In [ ]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
genai_client = genai.Client()

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
index.search('how to run ollama?')

In [ ]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=genai_client,
    instructions=instructions,
)

In [ ]:
assistant.rag("How do I run Ollama locally?")

In [ ]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

In [ ]:
messages = [
    {
        'role': 'user',
        'parts': [
            {
                'text': 'I just discovered the course. Can I join it?'
            }
        ]
    }
]

In [ ]:
response = genai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=messages,
)

response.text

In [ ]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )

In [ ]:
search("how do I run Ollama?")

Estruturação da função para chamada, a partir da implementação com o Gemini, que difere da OpenAI

In [ ]:
search_tool = {
    "function_declarations": [
        {
            "name": "search",
            "description": "Search the FAQ database for entries matching the given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                    }
                },
                "required": ["query"],
            }
        }
    ]
}

In [ ]:
response = genai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=messages,
    config={
        "tools": [search_tool]
    }
)

In [ ]:
call = response.candidates[0].content.parts[0].function_call

In [ ]:
args = response.candidates[0].content.parts[0].function_call.args

In [ ]:
results = search(**args)

In [ ]:
results

In [ ]:
response.candidates[0].content.parts

Até aqui, substituímos o que teria sido feita a implementação
diretamente por um desenvolvedor, para o LLM, para então fazer a busca em si dos resultados,
seguindo normalmente o fluxo do RAG

Isso pois a ideia é que um agente tenha memória, assim como ferramentas (tools) para interagir com o histórico

In [ ]:
import json

result_json = json.dumps(results, indent=2)
print(result_json)

In [ ]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.id,
    "output": result_json
}

Adicionar resposta do modelo com function call

In [ ]:
messages.append(response.candidates[0].content)

Adicionar resposta da tool

In [ ]:
messages.append({
    "role": "tool",
    "parts": [
        {
            "function_response": {
                "name": "search",
                "response": {
                    "result": result_json
                }
            }
        }
    ]
})

In [ ]:
response = genai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=messages,
    config={
        "tools": [search_tool]
    }
)

In [ ]:
response.text

In [ ]:
token_response = genai_client.models.count_tokens(
    model="gemini-3.5-flash",
    contents=messages,
)

print(f"Total input tokens: {token_response.total_tokens}")

In [ ]:
metadata = response.usage_metadata
print(f"Prompt Tokens: {metadata.prompt_token_count}")
print(f"Output Tokens: {metadata.candidates_token_count}")
print(f"Total Tokens: {metadata.total_token_count}")

In [ ]:
def calculate_gemini35flash_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 1.5
    OUTPUT_PRICE_PER_MILLION = 9.0

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gemini35flash_price(
    token_response.total_tokens, metadata.candidates_token_count
)
print("Total cost: $", round(result["total_cost"], 8))